In [1]:
import gradio as gr
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
import io
import wfdb
import os
import shutil
import tempfile
import pennylane as qml

# ==============================
# 🔹 QUANTUM SETUP
# ==============================
n_qubits = 4
n_qlayers = 3
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    inputs = torch.tanh(inputs) * torch.pi
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    for layer in range(n_qlayers):
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])

        for i in range(n_qubits):
            qml.RX(weights[layer, i, 0], wires=i)
            qml.RY(weights[layer, i, 1], wires=i)
            qml.RZ(weights[layer, i, 2], wires=i)

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# ==============================
# 🔹 MODEL (same as training)
# ==============================
class PatchEmbed(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, 192, kernel_size=16, stride=16)
        self.num_patches = (224 // 16) ** 2

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return F.layer_norm(x, (x.size(-1),))


class MSA(nn.Module):
    def __init__(self, dim=192, heads=4):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Sequential(nn.Linear(dim, dim))  # IMPORTANT

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)

        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(out)


class FeedForward(nn.Module):
    def __init__(self, dim=192):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(dim * 2, dim)
        )

    def forward(self, x):
        return self.fc(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(192)
        self.attn = MSA()
        self.norm2 = nn.LayerNorm(192)
        self.ff = FeedForward()

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


class QViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch = PatchEmbed()
        n = self.patch.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, 192))
        self.pos_emb = nn.Parameter(torch.zeros(1, n + 1, 192))

        self.blocks = nn.ModuleList([Block() for _ in range(6)])
        self.norm = nn.LayerNorm(192)

        self.proj = nn.Sequential(nn.Linear(192, n_qubits))  # IMPORTANT

        self.q_weights = nn.Parameter(
            torch.randn(n_qlayers, n_qubits, 3) * 0.1
        )

        self.head = nn.Sequential(
            nn.Linear(n_qubits, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        B = x.size(0)

        x = self.patch(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_emb[:, :x.size(1)]

        for blk in self.blocks:
            x = blk(x)

        x = self.norm(x)
        cls_token = x[:, 0]

        x_q = self.proj(cls_token)

        q_out = []
        for i in range(B):
            q = quantum_circuit(x_q[i], self.q_weights)
            q_out.append(torch.stack(q).float())

        q_out = torch.stack(q_out)
        return self.head(q_out)

# ==============================
# 🔹 LOAD MODEL
# ==============================
model = QViT()
model.load_state_dict(torch.load("ecg_qvit_final.pth", map_location="cpu", weights_only=True))
model.eval()

# ==============================
# 🔹 TRANSFORM (same as training)
# ==============================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ==============================
# 🔹 BLOCK 1: ECG → Spectrogram
# ==============================
def visualize_ecg(files):
    temp_dir = tempfile.mkdtemp()
    hea_path = None

    for file in files:
        fname = os.path.basename(file.name)
        dest = os.path.join(temp_dir, fname)
        shutil.copy(file.name, dest)

        if fname.endswith(".hea"):
            hea_path = dest

    record = wfdb.rdrecord(os.path.splitext(hea_path)[0])
    signal = record.p_signal[:, 0]

    # ECG plot
    plt.figure(figsize=(6,3))
    plt.plot(signal[:2000])
    plt.title("ECG Signal")
    buf1 = io.BytesIO()
    plt.savefig(buf1, format='png')
    plt.close()
    buf1.seek(0)

    # Spectrogram
    plt.figure(figsize=(4,4))
    plt.specgram(signal, Fs=360, NFFT=256, noverlap=128)
    plt.axis('off')
    buf2 = io.BytesIO()
    plt.savefig(buf2, format='png', bbox_inches='tight', pad_inches=0)
    plt.close()
    buf2.seek(0)

    return Image.open(buf1), Image.open(buf2)

# ==============================
# 🔹 BLOCK 2: Prediction
# ==============================
def predict_image(img):
    x = transform(img).unsqueeze(0)

    with torch.no_grad():
        out = model(x)
        probs = torch.softmax(out, dim=1)
        conf, pred = torch.max(probs, dim=1)

    classes = ["arrhythmia", "normal"]
    label = classes[pred.item()]

    return {
        "Arrhythmia": float(probs[0][0]),
        "Normal": float(probs[0][1])
    }, f"{label.upper()} ({conf.item()*100:.2f}%)"

# ==============================
# 🔹 UI
# ==============================
with gr.Blocks() as demo:
    gr.Markdown("# 🫀 QViT ECG Analysis System")

    # Block 1
    gr.Markdown("## 🔹 ECG to Spectrogram Visualization")
    file_input = gr.File(file_count="multiple")
    btn1 = gr.Button("Generate")

    ecg_img = gr.Image(label="ECG Signal")
    spec_img = gr.Image(label="Generated Spectrogram")

    btn1.click(visualize_ecg, file_input, [ecg_img, spec_img])

    # Block 2
    gr.Markdown("## 🔹 Spectrogram Classification")
    img_input = gr.Image(type="pil")
    btn2 = gr.Button("Predict")

    out_label = gr.Label()
    out_text = gr.Textbox(label="Prediction")

    btn2.click(predict_image, img_input, [out_label, out_text])

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
